# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SriyaHarshini19/PixelToPrediction/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule and its reason codes

Rule:
Prioritize pages that have not been updated for at least 180 days and have at least 500 impressions in the last 90 days. Among qualifying pages, higher-impression pages receive a higher priority score for refresh review.

Reason code:

stale_but_visible: The page is stale but still has meaningful search visibility, making it a candidate for refresh review.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# 2. Build the ranked queue

df["stale"] = df["days_since_last_update"] >= 180
df["visible"] = df["impressions_90d"] >= 500

df["score"] = (
    df["stale"].astype(int)
    * df["visible"].astype(int)
    * df["impressions_90d"]
)

df["reason_code"] = np.where(
    df["stale"] & df["visible"],
    "stale_but_visible",
    "not_selected"
)

df["action"] = np.where(
    df["stale"] & df["visible"],
    "review_refresh",
    "no_action"
)

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

ranked.head(10)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# 3. Top-20 review

top20 = ranked.head(20).copy()

top20["confidence_note"] = np.where(
    top20["score"] > 0,
    "High - meets stale and visibility thresholds",
    "Low - does not meet both thresholds"
)

top20["what_would_make_it_wrong"] = np.where(
    top20["score"] > 0,
    "The page may be intentionally unchanged or impressions may not represent valuable traffic.",
    "The page does not qualify as stale and visible."
)

top20[[
    "rank",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]]

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# 4. Weak picks + leakage check

weak_picks = ranked[ranked["score"] == 0].head(5)

print("Weak picks:")
display(weak_picks[["rank", "score", "reason_code", "action"]])

print("\nLeakage check:")
used_columns = ["days_since_last_update", "impressions_90d"]
print("Features used:", used_columns)
print("No product flags or future-window/label-derived inputs used.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.